# 2. Baseline modeling — 3D U-Net + transformer

Thin driver around the vendored baseline in `scripts/` (see
`docs/0_coding_standards.md` for why that logic lives in `scripts/` rather
than `src/` for now). Two independent things this notebook can do,
controlled by `RUN_MODE`:

- **`"submission"`**: predict on the real competition `test/` set and write
  `submission.csv`. Defaults to the baseline author's public pretrained
  checkpoint (`thibautgoldsborough/cellmot-baseline-artifacts`) so a first
  submission doesn't require training anything ourselves — see
  `docs/1_instructions.md`. **Run, validated, and submitted** — see the
  closing Findings cell.
- **`"train"`**: train our own checkpoint from scratch. Not yet run — a
  documented next-stage experiment (`docs/3_strategy.md`), not required
  for a first submission.

Runs on Kaggle via `scripts/push_kaggle_kernel.sh baseline` (competition
mount + the public `cellmot-baseline-artifacts` dataset, which bundles a
working `repo/` alongside pretrained `weights/` and offline dependency
`wheels/`, auto-detect — see the Setup cell), or point `$CELLMOT_DATA_DIR`
at a local copy. This is a **Code Competition**: the kernel runs with
internet disabled (`enable_internet: false`) since submissions must come
from a notebook rerun, not a file upload — see `docs/1_instructions.md`
for the full story.

Project docs (strategy, experiment log, submission history):
[`docs/`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/tree/main/docs).
This notebook's own markdown stays focused on what each section does and
found — the broader roadmap and full experiment/submission numbers live
there instead of being duplicated here.

## 1. Setup & Config

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )
    wheels_dir = str(ARTIFACTS_MOUNT / "wheels")

    # Code Competitions run with internet disabled, so every dependency is
    # installed offline from the artifacts dataset's bundled wheels/ rather
    # than PyPI/git. --no-deps keeps pip from touching numpy/scipy/llvmlite/
    # numba, which are already present and sufficient -- letting pip resolve
    # them normally corrupts numpy on this base image (see
    # docs/0_coding_standards.md's troubleshooting log for the full story).
    # polars is the one exception: it's present but too old for our code's
    # needs, and needs an explicit ==<version> pin to actually upgrade --
    # an unpinned name leaves an already-installed package untouched.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-index",
            "--find-links", wheels_dir, "--no-deps",
            "bidict", "donfig", "geff", "geff-spec", "ilpy", "imagecodecs",
            "numcodecs", "polars==1.42.0", "polars-runtime-32==1.42.0",
            "pyscipopt", "rustworkx", "tracksdata", "zarr",
        ],
        check=True,
    )

    # The dataset mounts read-only, but the vendored scripts write
    # predictions/weights relative to their own file location
    # (scripts/dataspec.py) -- copy the repo to a writable location first.
    REPO_ROOT = Path("/kaggle/working/repo")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(ARTIFACTS_MOUNT / "repo", REPO_ROOT)
else:
    REPO_ROOT = Path.cwd().parent
    ARTIFACTS_MOUNT = None  # pretrained weights are Kaggle-only; train locally instead

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

SEED = 0
RUN_MODE = "submission"  # "train" | "submission"

# --- "submission" mode: which weights to predict with ---------------------
USE_PRETRAINED = True  # True -> the public baseline checkpoint; False -> our own weights/ below
PRETRAINED_METHOD = "unet_transformer"
PRETRAINED_SPLIT = "0"

# --- "train" mode, and our-own-weights naming for "submission" mode -------
METHOD = "baseline"
SPLIT = "0"
EPOCHS = 3

# --- test-time detection/linking knobs (only used in "submission" mode) ---
# Back to 0.99 (the baseline author's own reported best): a local sweep
# found 0.90 predicted a gain (+0.0025 repaired score on 19 val videos),
# but the real submission at 0.90 scored 0.795 -- worse than both prior
# submissions (0.810, 0.817). Unlike the graph-repair fix (a structural
# correctness bug, verified transferable), DET_THRESHOLD directly controls
# detection volume, which interacts with the Adjusted Edge Jaccard's
# over-prediction penalty in a way that may not generalize from a small
# train-video sample to the real (and only 4-video) test set. See
# docs/3_strategy.md and docs/4_experiments.md for the full analysis.
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0

# --- graph repair (post-ILP), see docs/3_strategy.md ----------------------
# Physical voxel scale (Z, Y, X), microns/voxel -- confirmed in docs/2_eda_insights.md.
SCALE_ZYX = (1.625, 0.40625, 0.40625)
# Both ON by default: VALIDATE_ON_TRAIN_FOLD (19 val videos) confirmed both
# genuinely help once close_gaps' interpolated-node fix landed --
# CLOSE_GAPS alone +0.0020, PRUNE_SHORT_TRACKS alone +0.0024, combined
# +0.0065 edge_jaccard (0.8031 -> 0.8096) -- see section 2's finding and
# docs/3_strategy.md.
PRUNE_SHORT_TRACKS = True
PRUNE_MIN_NODES = 3        # drop connected components (tracks) with fewer nodes than this
CLOSE_GAPS = True
GAP_MAX = 2                # bridge dangling tracks missing up to this many consecutive frames
GAP_MAX_DIST_UM = 8.0      # max physical distance for a gap-closing match
GAP_MAX_ADDED_FRAC = 0.02  # cap new gap-closing bridges to this fraction of total nodes


def run(*args: str) -> None:
    """Run a vendored script with the current kernel's interpreter.

    Sets PYTHONPATH=<REPO_ROOT>/src so `import tracking_cellmot` resolves in
    the subprocess -- unlike a local `uv run`, nothing here `pip install -e`s
    the package, so it's only importable via sys.path/PYTHONPATH.
    """
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run([sys.executable, *args], check=True, cwd=REPO_ROOT, env=env)


def resolve_weights() -> tuple[Path, str]:
    """Return (checkpoint path, method name) per USE_PRETRAINED."""
    if USE_PRETRAINED:
        if ARTIFACTS_MOUNT is None:
            raise FileNotFoundError(
                "USE_PRETRAINED=True but the cellmot-baseline-artifacts dataset isn't "
                "mounted -- add it as a data source, or set USE_PRETRAINED=False to use "
                "our own weights/ (requires RUN_MODE='train' first)."
            )
        split_dir = ARTIFACTS_MOUNT / "weights" / PRETRAINED_METHOD / f"split_{PRETRAINED_SPLIT}"
        return split_dir / "edge_predictor_best.pth", PRETRAINED_METHOD
    split_dir = REPO_ROOT / "weights" / METHOD / f"split_{SPLIT}"
    return split_dir / "edge_predictor_best.pth", METHOD


print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}")
if IS_KAGGLE:
    print(f"ARTIFACTS_MOUNT={ARTIFACTS_MOUNT}")

## 2. Graph repair (post-ILP)

Per `docs/3_strategy.md`: every top-scoring public approach reviewed —
learned or classical — adds a deterministic repair stage after
detection/linking. Implemented here, both **off by default** — the
finding at the end of this section explains why, and `docs/3_strategy.md`
tracks the retuning plan.

- **Short-track pruning** (`PRUNE_SHORT_TRACKS`): drop connected components
  (tracks, including division lineages) with fewer than `PRUNE_MIN_NODES`
  nodes.
- **Bounded gap closing** (`CLOSE_GAPS`, up to `GAP_MAX` frames): a track
  that ends early (no outgoing edge, not at the last timepoint) is bridged
  to a track that starts late (no incoming edge, not at the first
  timepoint) `gap + 1` frames later, if within `GAP_MAX_DIST_UM` — a
  physical-space Hungarian assignment per timepoint, one gap size at a
  time (1-frame gaps closed before 2-frame, so a loose 2-frame setting
  can't introduce edges a correct 1-frame match would have caught first).
  Each bridge is expressed as `gap` interpolated intermediate nodes joined
  by ordinary single-frame edges, not one edge spanning multiple frames —
  see the finding below for why that distinction matters — and
  `GAP_MAX_ADDED_FRAC` caps how many bridges get added per pass, keeping
  only the cheapest.

Deferred for now (see `docs/3_strategy.md`'s roadmap): motion-aware
relinking (ILP already does global, flow-consistent linking, which is a
stronger starting point than the two-pass Hungarian these techniques
replace in detection-only public pipelines) and trajectory smoothing.

Unit-tested locally against synthetic data and a real `tracksdata` graph
object before ever touching Kaggle. Re-run `VALIDATE_ON_TRAIN_FOLD`
(section 6) after any retuning before re-enabling.

In [ ]:
import numpy as np
import polars as pl
import tracksdata as td
from scipy.optimize import linear_sum_assignment


def _connected_components(node_ids: list[int], edges: list[tuple[int, int]]) -> dict[int, int]:
    """Union-find over an edge list; returns node_id -> component root id."""
    parent = {n: n for n in node_ids}

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for s, t in edges:
        rs, rt = find(s), find(t)
        if rs != rt:
            parent[rs] = rt

    return {n: find(n) for n in node_ids}


def prune_short_tracks(nodes: pl.DataFrame, edges: pl.DataFrame, min_nodes: int) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Drop nodes/edges belonging to connected components with fewer than min_nodes nodes."""
    if min_nodes <= 1 or nodes.height == 0:
        return nodes, edges
    comp = _connected_components(
        nodes["node_id"].to_list(),
        list(zip(edges["source_id"].to_list(), edges["target_id"].to_list(), strict=True)),
    )
    comp_series = pl.Series("_comp", [comp[n] for n in nodes["node_id"]])
    sizes = comp_series.value_counts()
    keep_comps = set(sizes.filter(pl.col("count") >= min_nodes)["_comp"].to_list())
    keep_nodes = {n for n, c in comp.items() if c in keep_comps}
    kept_nodes = nodes.filter(pl.col("node_id").is_in(list(keep_nodes)))
    kept_edges = edges.filter(
        pl.col("source_id").is_in(list(keep_nodes)) & pl.col("target_id").is_in(list(keep_nodes))
    )
    return kept_nodes, kept_edges


def close_gaps(
    nodes: pl.DataFrame,
    edges: pl.DataFrame,
    scale_zyx: tuple[float, float, float],
    gap: int,
    max_dist_um: float,
    max_added_frac: float | None = None,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Bridge `gap`-frame-missing tracks via interpolated intermediate nodes.

    Ends at t are linked to starts at t+gap+1 through `gap` new nodes placed
    at evenly spaced positions between them -- one new node per skipped
    timepoint, joined by ordinary single-frame edges -- rather than one edge
    spanning multiple frames. Ground-truth edges only ever connect
    consecutive timepoints (`docs/metrics.md`), so a multi-frame edge can
    never match one; every bridge needs the intermediate nodes to have any
    chance of being scored correctly. `max_added_frac`, if set, keeps only
    the cheapest (lowest-cost) bridges up to that fraction of total nodes,
    matching the reference notebook's rate limiter (`docs/3_strategy.md`).
    """
    if nodes.height == 0:
        return nodes, edges
    out_ids = set(edges["source_id"].to_list())
    in_ids = set(edges["target_id"].to_list())
    t_min, t_max = int(nodes["t"].min()), int(nodes["t"].max())

    ends = nodes.filter(~pl.col("node_id").is_in(list(out_ids)) & (pl.col("t") < t_max))
    starts = nodes.filter(~pl.col("node_id").is_in(list(in_ids)) & (pl.col("t") > t_min))

    scale = np.array(scale_zyx)
    candidates = []  # (cost, t, e_id, s_id, e_zyx, s_zyx)
    used_starts: set[int] = set()
    for t in sorted(ends["t"].unique().to_list()):
        e_t = ends.filter(pl.col("t") == t)
        s_t = starts.filter((pl.col("t") == t + gap + 1) & (~pl.col("node_id").is_in(list(used_starts))))
        if e_t.height == 0 or s_t.height == 0:
            continue
        e_zyx = e_t.select(["z", "y", "x"]).to_numpy()
        s_zyx = s_t.select(["z", "y", "x"]).to_numpy()
        cost = np.linalg.norm((e_zyx[:, None, :] - s_zyx[None, :, :]) * scale, axis=2)
        ri, ci = linear_sum_assignment(cost)
        e_ids = e_t["node_id"].to_list()
        s_ids = s_t["node_id"].to_list()
        for r, c in zip(ri, ci, strict=True):
            if cost[r, c] > max_dist_um:
                continue
            s_id = int(s_ids[int(c)])
            if s_id in used_starts:
                continue
            candidates.append((cost[r, c], t, int(e_ids[int(r)]), s_id, e_zyx[r], s_zyx[c]))
            used_starts.add(s_id)

    if not candidates:
        return nodes, edges

    candidates.sort(key=lambda c: c[0])
    if max_added_frac is not None:
        cap = max(1, round(nodes.height * max_added_frac))
        candidates = candidates[:cap]

    next_id = int(nodes["node_id"].max()) + 1
    new_nodes = []
    new_edges = []
    for _, t, e_id, s_id, e_zyx, s_zyx in candidates:
        prev_id = e_id
        for k in range(1, gap + 1):
            frac = k / (gap + 1)
            pos = e_zyx + (s_zyx - e_zyx) * frac
            new_id = next_id
            next_id += 1
            new_nodes.append(
                {"node_id": new_id, "t": t + k, "z": float(pos[0]), "y": float(pos[1]), "x": float(pos[2])}
            )
            new_edges.append({"source_id": prev_id, "target_id": new_id})
            prev_id = new_id
        new_edges.append({"source_id": prev_id, "target_id": s_id})

    nodes_out = pl.concat([nodes, pl.DataFrame(new_nodes, schema=nodes.schema)])
    edges_out = pl.concat([edges, pl.DataFrame(new_edges, schema=edges.schema)])
    return nodes_out, edges_out


def repair_graph(graph: "td.graph.BaseGraph") -> "td.graph.BaseGraph":
    """Apply gap-closing then short-track pruning to a predicted graph; returns a fresh graph.

    Reads the PRUNE_*/CLOSE_GAPS/GAP_*/SCALE_ZYX config above. Returns a new
    tracksdata InMemoryGraph -- the input graph is not mutated.
    """
    nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])
    edges = graph.edge_attrs(attr_keys=["source_id", "target_id"])

    if CLOSE_GAPS:
        for g in range(1, GAP_MAX + 1):
            nodes, edges = close_gaps(
                nodes, edges, SCALE_ZYX, gap=g, max_dist_um=GAP_MAX_DIST_UM, max_added_frac=GAP_MAX_ADDED_FRAC
            )

    if PRUNE_SHORT_TRACKS:
        nodes, edges = prune_short_tracks(nodes, edges, PRUNE_MIN_NODES)

    out = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        out.add_node_attr_key(key, pl.Float64, 0.0)
    id_map: dict[int, int] = {}
    for row in nodes.iter_rows(named=True):
        new_id = out.add_node({"t": int(row["t"]), "z": float(row["z"]), "y": float(row["y"]), "x": float(row["x"])})
        id_map[row["node_id"]] = new_id
    for row in edges.iter_rows(named=True):
        s, t = id_map.get(row["source_id"]), id_map.get(row["target_id"])
        if s is not None and t is not None:
            out.add_edge(s, t, {})
    return out

**Finding:** graph repair initially *regressed* the score (edge_jaccard
0.8031 → 0.7897) — a first implementation bridged dangling track ends with
a single edge spanning multiple frames, which no ground-truth edge (always
`t → t+1`) can ever match. Fixed by inserting interpolated intermediate
nodes per bridge instead (the code above); re-validated, both
`CLOSE_GAPS` and `PRUNE_SHORT_TRACKS` turned out to genuinely help on
their own, and more than additively combined
(+0.0065 edge_jaccard) — the original regression was entirely the
gap-closing bug, not a real problem with either technique. Both are on by
default now, and this config has been submitted.

Full experiment-by-experiment numbers and root-cause detail:
[`docs/4_experiments.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/4_experiments.md).

## 3. Train (optional — skip if `USE_PRETRAINED`)

Only runs in `RUN_MODE == "train"`. Not needed for a first submission (see
`USE_PRETRAINED` above) — this is how to train our own checkpoint to try to
beat the public baseline later.

In [ ]:
if RUN_MODE == "train":
    run(
        "scripts/train_unet_transformer.py",
        "--split", SPLIT,
        "--epochs", str(EPOCHS),
    )
    print(f"Trained weights: {REPO_ROOT}/weights/{METHOD}/split_{SPLIT}/edge_predictor_best.pth")

*Not yet run — `RUN_MODE` has stayed `"submission"` so far, since the
pretrained checkpoint gets a real result faster than training from
scratch (`docs/3_strategy.md`'s roadmap). Once it runs, this cell should
report the training loss curve, whether it converged within `EPOCHS`
epochs, and any stability issues.*

## 4. Predict on the competition test set

Only runs in `RUN_MODE == "submission"`. `predict_unet_transformer.py`
requires a `dataset_splits.json` listing which videos to predict — the real
`test/` directory doesn't ship one (that's a train-only, fold-splitting
concept), so build a synthetic one-fold file listing every test video
first, matching the approach in the baseline author's own public inference
notebook (`thibautgoldsborough/unet-baseline-inference-submission`).

In [ ]:
if RUN_MODE == "submission":
    import json

    test_stems = sorted(p.stem for p in TEST_DIR.glob("*.zarr"))
    print(f"{len(test_stems)} test videos under {TEST_DIR}")

    test_splits_file = REPO_ROOT / "kaggle_test_splits.json"
    test_splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}]))

    weights_path, predict_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(TEST_DIR),
        "--splits", str(test_splits_file),
        "--split", "0",
        "--method", predict_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

## 5. Build `submission.csv`

Applies graph repair (section 2) to each predicted `.geff`, then flattens
the repaired graphs into the competition's CSV schema — verified against
the real `sample_submission.csv` downloaded via the Kaggle CLI:
`id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`, one `node` row
per detection and one `edge` row per link.

Flattening is inlined rather than calling `scripts/geffs_to_csv.py`: the
artifacts dataset's bundled `repo/` only includes what the baseline
author's own inference notebook needs (train/predict/dataspec), not this
project's extra conversion scripts (`geffs_to_csv.py`, `csv_to_geffs.py`,
`evaluate.py`) — confirmed missing on a real run. This mirrors the exact
logic in `scripts/geffs_to_csv.py`, kept in sync by hand for now.

In [ ]:
if RUN_MODE == "submission":
    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    predictions_dir = REPO_ROOT / "predictions" / kaggle_user / predict_method / "split_0"
    submission_csv = Path("/kaggle/working/submission.csv") if IS_KAGGLE else REPO_ROOT / "submission.csv"

    def _graph_to_rows(graph, name: str) -> pl.DataFrame:
        """Flatten one graph into node rows then edge rows (submission schema)."""
        nodes = graph.node_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("node").alias("row_type"),
            pl.col("node_id").cast(pl.Int64),
            pl.col("t").cast(pl.Int64),
            pl.col("z").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("y").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("x").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.lit(-1, dtype=pl.Int64).alias("source_id"),
            pl.lit(-1, dtype=pl.Int64).alias("target_id"),
        )
        edges = graph.edge_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("edge").alias("row_type"),
            pl.lit(-1, dtype=pl.Int64).alias("node_id"),
            pl.lit(-1, dtype=pl.Int64).alias("t"),
            pl.lit(-1, dtype=pl.Int64).alias("z"),
            pl.lit(-1, dtype=pl.Int64).alias("y"),
            pl.lit(-1, dtype=pl.Int64).alias("x"),
            pl.col("source_id").cast(pl.Int64),
            pl.col("target_id").cast(pl.Int64),
        )
        return pl.concat([nodes, edges])

    geffs = sorted(predictions_dir.glob("*.geff"))
    frames = []
    for g in geffs:
        graph = td.graph.IndexedRXGraph.from_geff(str(g))
        graph = graph[0] if isinstance(graph, tuple) else graph
        n_before, e_before = graph.num_nodes(), graph.num_edges()
        graph = repair_graph(graph)
        frames.append(_graph_to_rows(graph, g.stem))
        print(
            f"{g.stem}: {n_before} nodes, {e_before} edges "
            f"-> repaired: {graph.num_nodes()} nodes, {graph.num_edges()} edges"
        )

    columns = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    table = pl.concat(frames) if frames else pl.DataFrame(schema=dict.fromkeys(columns, pl.Int64))
    table = table.with_row_index("id")
    table.write_csv(submission_csv)
    print(f"Wrote {table.height} rows from {len(geffs)} geffs to {submission_csv}")

## 6. (Optional) Validate methodology on a train fold

The real `test/` set has no local ground truth to score against. To
sanity-check the weights/detection/linking/**repair** config *before*
spending a submission attempt, predict on a held-out **train** fold instead
(real GT available) and score locally with the competition's own metric
(`docs/1_instructions.md` / `docs/metrics.md`) — both **with and without
graph repair**, so the repair stage's actual effect is visible before it's
trusted. Builds its own deterministic 90/10 train/val split the same way
`train_unet_transformer.py` does when no `dataset_splits.json` is present,
so it doesn't depend on a prior run.

Also sweeps `DET_THRESHOLD` across `DET_THRESHOLD_CANDIDATES` on the same
val split. **Caution**: a prior sweep here predicted `DET_THRESHOLD=0.90`
as an improvement, but the real submission regressed sharply (0.795 vs
0.817) — see the closing insight cell and `docs/4_experiments.md` for the
full analysis before trusting this sweep's "best" value without a
confirming submission.

In [ ]:
VALIDATE_ON_TRAIN_FOLD = False  # set True to sanity-check before submitting
DET_THRESHOLD_CANDIDATES = [0.90, 0.95, 0.99, 0.995]  # 0.90 is the current default -- see section 6 intro

if VALIDATE_ON_TRAIN_FOLD:
    import json
    import random

    from tracking_cellmot.io import open_dataset
    from tracking_cellmot.metrics import evaluate_datasets

    stems = sorted(
        p.name[:-5] for p in DATASET_PATH.glob("*.zarr")
        if (DATASET_PATH / f"{p.name[:-5]}.geff").exists()
    )
    random.Random(0).shuffle(stems)
    n_val = max(1, len(stems) // 10)
    val_stems = stems[:n_val]
    train_splits_file = REPO_ROOT / "kaggle_train_splits.json"
    train_splits_file.write_text(json.dumps(
        [{"split": 0, "train": stems[n_val:], "test": val_stems}]
    ))
    print(f"{len(stems) - n_val} train / {n_val} val videos under {DATASET_PATH}")

    weights_path, validate_method = resolve_weights()
    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    val_predictions_dir = REPO_ROOT / "predictions" / kaggle_user / validate_method / "split_0"

    def _predict_and_score(det_threshold: float) -> tuple:
        """Run predict at det_threshold on the val split, return (raw, repaired) evaluate_datasets results."""
        predict_args = [
            "scripts/predict_unet_transformer.py",
            "--data-dir", str(DATASET_PATH),
            "--splits", str(train_splits_file),
            "--split", "0",
            "--method", validate_method,
            "--weights", str(weights_path),
            "--unet-batch-size", str(UNET_BATCH_SIZE),
            "--det-threshold", str(det_threshold),
            "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
            "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
            "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
            "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
        ]
        if USE_ILP:
            predict_args.append("--use-ilp")
        run(*predict_args)  # writes one .geff per val video -- no --evaluate, we score below ourselves

        raw_pairs, repaired_pairs = [], []
        for stem in val_stems:
            geff_path = val_predictions_dir / f"{stem}.geff"
            if not geff_path.exists():
                print(f"  {stem}: no prediction found, skipped")
                continue
            pred_graph = td.graph.IndexedRXGraph.from_geff(str(geff_path))
            pred_graph = pred_graph[0] if isinstance(pred_graph, tuple) else pred_graph
            gt_graph = open_dataset(
                DATASET_PATH / stem, normalize=False, load_image=False, require_tracks=True
            ).tracks
            raw_pairs.append((pred_graph, gt_graph))
            repaired_pairs.append((repair_graph(pred_graph), gt_graph))
        return evaluate_datasets(raw_pairs, scale=SCALE_ZYX), evaluate_datasets(repaired_pairs, scale=SCALE_ZYX)

    sweep_results = []
    for det_threshold in DET_THRESHOLD_CANDIDATES:
        print(f"\n--- DET_THRESHOLD={det_threshold} ---")
        raw_result, repaired_result = _predict_and_score(det_threshold)
        sweep_results.append((det_threshold, raw_result, repaired_result))
        print(
            f"  raw:      edge_jaccard={raw_result.edge_jaccard:.4f}  "
            f"division_jaccard={raw_result.division_jaccard:.4f}  score={raw_result.score:.4f}"
        )
        print(
            f"  repaired: edge_jaccard={repaired_result.edge_jaccard:.4f}  "
            f"division_jaccard={repaired_result.division_jaccard:.4f}  score={repaired_result.score:.4f}"
        )

    header = f"{'DET_THRESHOLD':>14s} {'raw score':>12s} {'repaired score':>16s} {'best':>8s}"
    print(f"\n{header}")
    for det_threshold, raw_result, repaired_result in sweep_results:
        best = "repaired" if repaired_result.score >= raw_result.score else "raw"
        print(f"{det_threshold:14.3f} {raw_result.score:12.4f} {repaired_result.score:16.4f} {best:>8s}")
    best_threshold, _, best_repaired = max(sweep_results, key=lambda r: r[2].score)
    print(f"\nBest: DET_THRESHOLD={best_threshold}, repaired score={best_repaired.score:.4f}")

*Insight — a real miss, not just a regression: `DET_THRESHOLD=0.90`
predicted a gain here (repaired score 0.8121 vs 0.8096 at 0.99, +0.0025),
but the real submission scored **0.795** — worse than 0.99's confirmed
0.817 by -0.022, and worse even than the original repair-off submission
(0.810). Reverted to `DET_THRESHOLD=0.99` above. Leading hypotheses (not
yet distinguished): (1) picking the empirical best of several candidates
on a small 19-video sample is a genuine multiple-comparisons risk, unlike
the repair fix, which was one hypothesis-driven test, not a search over
candidates; (2) `DET_THRESHOLD` directly controls detection volume, which
interacts with the Adjusted Edge Jaccard's over-prediction penalty — a
term that may behave very differently on the real (and only 4-video) test
set than on a random 19-video train sample, especially given
`01_eda.ipynb`'s own finding of ~15× annotation-density variance across
videos. See `docs/3_strategy.md` and `docs/4_experiments.md`.*

## Findings / limitations / next experiment

- **Findings**: predict + submission pipeline completes end-to-end on
  Kaggle GPU (`device=cuda`, `NvidiaTeslaT4`), internet disabled, using
  the pretrained `unet_transformer` checkpoint. Full submission history:
  [`docs/5_submissions.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/5_submissions.md).
  Best confirmed config: repair on, `DET_THRESHOLD=0.99`, **public
  leaderboard 0.817**. A `DET_THRESHOLD=0.90` sweep predicted a further
  gain locally but regressed sharply on the real leaderboard (0.795) —
  the first case this session where `VALIDATE_ON_TRAIN_FOLD` didn't
  transfer, after two prior cases (the repair fix, twice) where it did.
  Full experiment log and analysis:
  [`docs/4_experiments.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/4_experiments.md).
- **Limitations**: the pretrained checkpoint (`unet_transformer`, split 0)
  wasn't trained to convergence per the baseline author's own notes.
  `VALIDATE_ON_TRAIN_FOLD` is reliable for single, hypothesis-driven
  changes (confirmed twice) but its reliability for *selecting* among
  several candidate values (the `DET_THRESHOLD` sweep) is now in
  question — a real submission is needed to confirm any future threshold
  or `ILP_*_WEIGHT` change before trusting it. Division recovery hasn't
  been attempted — `01_eda.ipynb`'s wider survey found only 1 division
  across 1,000 combined timepoints, suggesting genuine rarity rather than
  an under-detection issue, but this isn't confirmed.
- **Next**: full prioritized roadmap in
  [`docs/3_strategy.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/3_strategy.md) —
  currently investigating why the `DET_THRESHOLD` sweep didn't transfer
  before trying any further hyperparameter changes.